# Day 3: Batch Ingestion (X12 EDI Claims & Debezium CDC)

This notebook demonstrates processing synthetic raw EDI lines and Debezium JSON records into structured Bronze tables.

In [ ]:
import json
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, split, monotonically_increasing_id, current_timestamp, lit, to_json
from pyspark.sql.types import StructType, StructField, StringType, MapType

spark = SparkSession.builder \
    .appName("Day03_Batch_CDC") \
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.4_2.12:1.4.1") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type", "hadoop") \
    .config("spark.sql.catalog.local.warehouse", "/tmp/iceberg_warehouse") \
    .getOrCreate()


In [ ]:
spark.sql("""
CREATE DATABASE IF NOT EXISTS local.ehdip_bronze_db
""")
spark.sql("""
CREATE TABLE IF NOT EXISTS local.ehdip_bronze_db.raw_payloads (
    ingestion_timestamp TIMESTAMP,
    source_system_id STRING,
    payload_id STRING,
    raw_payload_json STRING
)
USING iceberg
PARTITIONED BY (days(ingestion_timestamp))
""")

## 1. Process X12 EDI Simulator

In [ ]:
# Simulate X12 payload
x12_data = [("ISA*00*...~GS*HC*...~ST*837*...~",)]
x12_df = spark.createDataFrame(x12_data, ["value"])

parsed_x12 = x12_df.withColumn("payload_id", monotonically_increasing_id().cast("string")) \
    .withColumn("raw_payload_json", col("value")) \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("source_system_id", lit("x12_batch")) \
    .select("ingestion_timestamp", "source_system_id", "payload_id", "raw_payload_json")

parsed_x12.write.format("iceberg").mode("append").saveAsTable("local.ehdip_bronze_db.raw_payloads")
print("X12 appended.")

## 2. Process Debezium CDC Simulator

In [ ]:
# Simulate Debezium JSON payload
cdc_data = [{
    "op": "c",
    "after": {"id": "pat_123", "name": "John Doe", "status": "active"},
    "before": None,
    "source": {"db": "postgres"}
}]

schema = StructType([
    StructField("op", StringType(), True),
    StructField("after", MapType(StringType(), StringType()), True),
    StructField("before", MapType(StringType(), StringType()), True),
    StructField("source", MapType(StringType(), StringType()), True)
])

import json
cdc_json_rdd = spark.sparkContext.parallelize([json.dumps(r) for r in cdc_data])
cdc_df = spark.read.json(cdc_json_rdd, schema=schema)

parsed_cdc = cdc_df.filter(col("op").isin("c", "u", "d")) \
    .withColumn("payload_id", col("after.id")) \
    .withColumn("raw_payload_json", to_json(col("after"))) \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("source_system_id", lit("debezium_cdc")) \
    .select("ingestion_timestamp", "source_system_id", "payload_id", "raw_payload_json")

parsed_cdc.write.format("iceberg").mode("append").saveAsTable("local.ehdip_bronze_db.raw_payloads")
print("CDC appended.")

In [ ]:
# Verify Bronze table
spark.sql("SELECT * FROM local.ehdip_bronze_db.raw_payloads").show(truncate=False)